# Paso 2 — Flood History + Water-derived Topography
Sentinel-1 SAR 2014-2024 → flood masks → frequency raster → waterlines → fill order.
Gate: comparar topografía-desde-agua vs DEM.

In [ ]:
import sys
sys.path.insert(0, '..')
from src.config import load_config, get_bbox
from src.gee import init_gee, get_chirps_events, get_s1_full_series, export_to_drive
from src.flood import build_flood_inventory, compute_flood_frequency
from src.water_topo import frequency_to_relative_elevation, compare_dem_vs_water_topo
from src.viz import plot_raster
from pathlib import Path

cfg = load_config()
init_gee()

## 2.1 Get rain events from CHIRPS

In [ ]:
bbox_ee = get_bbox(cfg, as_ee=True)
event_dates = get_chirps_events(
    bbox_ee,
    start=cfg['sar']['start_date'],
    end=cfg['sar']['end_date'],
    threshold_mm=cfg['chirps']['event_threshold_mm'],
)
print(f'{len(event_dates)} events found')
print(event_dates[:5])

## 2.2 Build flood inventory (GEE)

In [ ]:
inventory = build_flood_inventory(
    bbox_ee, event_dates, output_dir=None,
    polarization=cfg['sar']['polarization'],
)
flood_freq = compute_flood_frequency(inventory)

# Export flood frequency to Drive → download manually to data/raw/
export_to_drive(flood_freq, 'flood_frequency', bbox_ee, scale=30)

## 2.3 Water-derived topography
After downloading flood_frequency.tif from Drive to data/raw/:

In [ ]:
processed_dir = Path('../data/processed')
raw_dir = Path('../data/raw')

rel_elev = frequency_to_relative_elevation(
    flood_freq_path=raw_dir / 'flood_frequency.tif',
    output_path=processed_dir / 'water_rel_elev.tif',
)
plot_raster(processed_dir / 'water_rel_elev.tif',
            title='Relative Elevation (from flood freq)', cmap='terrain',
            output_path='../outputs/water_rel_elev.png')

## 2.4 Gate: DEM vs water-topo validation
Spearman r < 0.4 → use water topo as primary elevation input.

In [ ]:
corr = compare_dem_vs_water_topo(
    dem_path=processed_dir / 'pit_filled.tif',
    rel_elev_path=processed_dir / 'water_rel_elev.tif',
    output_path=processed_dir / 'dem_water_discrepancy.tif',
)
if corr < 0.4:
    print('ACTION: Use water_rel_elev.tif instead of DEM for elevation feature in ML.')
else:
    print('DEM and water topo agree. Both will be used as features.')